In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ast

df = pd.read_csv('rj_listings.csv') #RJ como cidade escolhida

In [ ]:
print(df.head())


In [ ]:
df.columns.to_list()

In [ ]:
df.info()

In [ ]:
#Colunas que possuem nulos
nulos_por_coluna = df.isnull().sum()
print(nulos_por_coluna[nulos_por_coluna > 0])

In [ ]:
df['price'].dtype #Checar o tipo da coluna

In [ ]:
#remove $ para transformar posteriormente em float
df['price'] = df['price'].str.replace('$', '', regex=False) 
#remove , para transformar posteriormente em float
df['price'] = df['price'].str.replace(',', '', regex=False)

In [ ]:
#Transforma em Float
df['price'] = df['price'].astype(float)

In [ ]:
df['price'].describe()

In [ ]:
df['amenities'].describe()

In [ ]:
#Descobre as top 5 amenidades para criar categorias binarias
pd.Series([item for sublist in df['amenities'].dropna().apply(ast.literal_eval) for item in sublist]).value_counts().head(5)

As cinco amenidades escolhidas foram Wifi, Kitchen, TV, Hot water e Air conditioning, pois apresentaram as maiores frequências na base. Dessa forma, a análise concentra-se nas comodidades mais comuns entre os anúncios da cidade.

In [ ]:
top_amenidades = [
    'Wifi',
    'Kitchen',
    'TV',
    'Hot water',
    'Air conditioning'
] #Cria uma coluna para cada uma das 5 maiores amenidades

for amenidade in top_amenidades:
    df[amenidade] = df['amenities'].apply(
        lambda x: 1 if amenidade in x else 0
    )

In [ ]:
df[top_amenidades].head()

Pergunta 1 a ser respondida: Quais amenidades estão mais associadas ao preço das acomodações?

In [ ]:
#para cada uma das 5 amenidades, o preço mediano dos anúncios que possuem a 
#amenidade com o preço mediano dos anúncios que não possuem.
for amenidade in top_amenidades:
    results = df.groupby(amenidade)['price'].agg(
        ['count', 'mean', 'median']
    )

    print(f'\n {amenidade} ')
    print(results)

Quanto a mediana do preco muda comparando acomodacoes que tem determinada amenidade com as que nao tem.

In [ ]:
#Quanto o preço mediano varia, em termos percentuais, entre acomodações que 
#possuem determinada amenidade e acomodações que não possuem?
percentuais = {}

for amenidade in top_amenidades:
    results = df.groupby(amenidade)['price'].agg(
        ['count', 'mean', 'median']
    )

    mediana_sem = results.loc[0, 'median']
    mediana_com = results.loc[1, 'median']

    percentuais[amenidade] = (
        (mediana_com - mediana_sem) / mediana_sem
    ) * 100

percentuais = pd.Series(percentuais).sort_values(ascending=False)

percentuais

In [ ]:
percentuais.plot(kind='bar', color='purple')

plt.xlabel('Amenidade')
plt.ylabel('Diferença percentual da mediana (%)')
plt.title('Diferença percentual do preço mediano por amenidade')

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

Análise aprofundada: A associação entre as amenidades e o preço das acomodações permanece quando controlamos por características da acomodação?

In [ ]:
#Comparação entre acomodações com e sem TV
df['has_tv'] = df['amenities'].str.lower().str.contains('tv', na=False)
df['has_tv'] = df['has_tv'].map({True: 'Com TV', False: 'Sem TV'})
tabela_preco= pd.pivot_table(
    df,
    values='price', 
    index=['bedrooms'],
    columns=['has_tv'], 
    aggfunc='median' 
).round(2)
tabela_preco.index = tabela_preco.index.astype(int)
tabela_preco['Diferença (R$)'] = tabela_preco['Com TV'] - tabela_preco['Sem TV']
tabela_preco = tabela_preco.fillna('-')

tabela_volume = pd.pivot_table(
    df,
    values='price', 
    index=['bedrooms'],
    columns=['has_tv'],
    aggfunc='count' 
).fillna(0)
tabela_volume.index = tabela_volume.index.astype(int)
display(tabela_preco.head(7)) 
display(tabela_volume.head(7))

Pode se notar que, excluindo os estudios(bedrooms=0), há uma disparidade de preço meio inconsistente comparando exemplos com e sem tv.   
Vale resaltar que a quantidade de locais sem tv é muito abaixo da de locais com tv.   


Locais com apenas um quarto são os que possuem mais dados, onde a mediana retorna que há uma diferença de R$99 reais entre quartos com e sem TV.   
De dois a quatro quartos, a presença da TV afeta menos o preço, considerando que quartos a mais já adicionam um bom peso ao preço.   

Os ambientes com 5+ quartos são questionaveis a analise, pode se extrair que a falta de uma TV indica que outras qualidades também não estão presentes, levando a grande diferença de preço, porem devido a quantidade de dados, não pode se extrair muito mais do que isto.

In [ ]:
tabela_grafico = pd.pivot_table(
    df[(df['bedrooms'] >= 1) & (df['bedrooms'] <= 4)],
    values='price', 
    index=['bedrooms'],
    columns=['has_tv'], 
    aggfunc='median' 
)
tabela_grafico.index = tabela_grafico.index.astype(int)
ax = tabela_grafico.plot(kind='bar', width=0.5,color=["#28a3c9","#c94d28"])
ax.set_facecolor('#eeeeee') 
ax.figure.patch.set_facecolor('#eeeeee')
sns.despine(top=True, right=True)
plt.title('Preço mediano por numero de quartos', fontsize=14)
plt.xlabel('Numero de quartos', fontsize=14)
plt.ylabel('Preço mediano (R$)', fontsize=14)
plt.legend(title='Local')
plt.xticks(rotation=0)
plt.show()

Conseguimos evidenciar melhor a discrepancia de preço e dar luz ao que foi abordado na tabela anterior, elminando estudios e locais com mais de 4 quartos, é possivel ver uma crescente estável que mantém os padrões de locais sem tv serem mais baratos.

In [ ]:
lista_amenidades = ['tv', 'kitchen', 'air conditioning']
nome_pacote = 'todas amenidades'

df_temp = df.copy()
condicao_combo = pd.Series(True, index=df_temp.index)

for amenidade in lista_amenidades:
    tem = df_temp['amenities'].str.lower().str.contains(amenidade.lower(), na=False)
    condicao_combo = condicao_combo & tem

nome_coluna = 'Amenidades'
df_temp[nome_coluna] = condicao_combo.map({
    True: f'Contém {nome_pacote}', 
    False: f'Não contém {nome_pacote}'
})

df_grafico = df_temp[
    (df_temp['bedrooms'] >= 1) &     
    (df_temp['bathrooms'] >= 1) &              
    (df_temp['room_type'] != 'Hotel room') &   
    (df_temp['room_type'] != 'Shared room') &    
    (df_temp['bedrooms'] <= 4) & 
    (df_temp['bathrooms'] <= 3)
]
df_agrupado = df_grafico.groupby(['bedrooms', 'bathrooms', nome_coluna])['price'].median().reset_index()
df_agrupado = df_agrupado.rename(columns={'bathrooms': 'Banheiros'})

plt.figure(figsize=(10, 6), dpi=300)
plt.gca().set_facecolor('#eeeeee')
plt.gcf().patch.set_facecolor('#eeeeee')

ax = sns.scatterplot(
    data=df_agrupado,
    x='bedrooms',
    y='price',
    size='Banheiros',
    sizes=(200, 1000),
    hue=nome_coluna,
    palette=["#28a3c9","#c94d28"], 
    alpha=0.8,
)
sns.despine(top=True, right=True)

plt.title('Preço Mediano entre locais com e sem amenidades', fontsize=14, fontweight='bold', loc='center', pad=30)
plt.xlabel("Número de Quartos", fontsize=14)
plt.ylabel("Preço Mediano (R$)", fontsize=14)
plt.tick_params(axis='both', labelsize=10) 
plt.xticks([1, 2, 3, 4])
plt.legend(
    bbox_to_anchor=(1.05, 0.7), 
    loc='upper left', frameon=True, 
    facecolor='#fdfdfd', 
    labelspacing=1.0, 
    title="legenda",
    prop={'size': 10},
    title_fontproperties={'weight': 'bold', 'size': 12}
)
plt.tight_layout()
plt.show()

Embora o gráfico de barras sejá muito bom para visualização, o scatterplot apresenta mais dimensões para comparação, aumentando os fatores para a análise. 
Pode-se observar a disparidade criada ao trazer o número de banheiros a comparação. É possível identificar que ambientes com 1 quarto seguem a hiptese

Porém, pode se observar que ao progredir pelo número de quartos, a diferença entre os preços cai. Ainda sim, o grafico mosra que os locais com as 3 maiores amenidades presentes ainda lideram nos preços. Vale notar que ambientes com 4 quartos começam a deturpar o padrão. Isso se deve ao impacto que cada local tem na mediana e à quantidade menor de dados com quartos e banheiros elevados.

Após estes fatores, pode se concluir que a associação entre as amenidades e o preço das acomodações permanece ao analizar diferentes locais. Com um baixo número de quartos e banheiros, pode-se ver que o preço aumenta mais consistente quando tem amenidades. Porém, ao escalar o número de banheiros e quartos, há uma queda nesta consistencia.